## 四种状态使用案例 全局、输入、输出、私有状态

In [2]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


# 1、状态定义
# 1.1 输入状态
class InputState(TypedDict):
    username: str


# 1.2 输出状态
class OutputState(TypedDict):
    graph_output: str


# 1.3 全局状态（必须包含输入、输出状态）
class OverAllState(TypedDict):
    username: str
    graph_output: str
    nickname: str


# 1.4 私有状态
class PrivateState(TypedDict):
    greeting: str


# 2、定义节点
# 2.1 第一个节点 对接start => InputState 修改的内容在全局状态中=>OverAllState
def node_1(state: InputState) -> OverAllState:
    # 向全局状态添加username
    return {
        "nickname": "Dear " + state["username"]
    }


# 2.2 第二个节点，对接node_1 => OverallState 使用的参数在全局状态中 修改的参数在私有状态中 -> PrivateState
def node_2(state: OverAllState) -> PrivateState:
    # 向私有状态中添加greeting
    return {
        "greeting": "Hello, " + state["nickname"]
    }


# 2.2 第二个节点，对接node_2 => PrivateState 使用的参数在私有状态中 修改的参数在输出状态中 -> OutputState
def node_3(state: PrivateState) -> OutputState:
    # 向输出状态中添加graph_output
    return {
        "graph_output": state["greeting"] + " 很高兴认识你！"
    }


# 3、创建状态图
# 3.1 定义图的时候 加载全局状态 输入状态 输出状态
builder = StateGraph(state_schema=OverAllState, input_schema=InputState, output_schema=OutputState)
# 3.2 添加节点
# 添加节点的时候添加私有状态
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)
# 3.3 添加边
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", "node_3")
builder.add_edge("node_3", END)
# 获取图
graph = builder.compile()
# 调用图，传入输入状态
result = graph.invoke({"username": "Sevan"})
# 返回： 输出状态，不再是全局状态
print(result)

{'graph_output': 'Hello, Dear Sevan 很高兴认识你！'}
